# Python 解包（Unpacking）完全教程

> **一句话本质**：解包是"把容器拆成独立变量，或把独立值组装成参数"的双向工具——`*` 管位置，`**` 管关键字。

## 学习目标
1. 掌握基本解包、扩展解包（`*` 收集）、嵌套解包的语法与底层迭代器协议
2. 理解函数定义端（`*args` / `**kwargs`）与调用端（`f(*a, **kw)`）的**对称关系**
3. 熟练使用 `{**d1, **d2}`、`[*a, *b]` 完成容器合并与配置覆盖
4. 识别解包的典型陷阱（数量不匹配、生成器耗尽、`**` 键必须是字符串、嵌套结构不匹配等）并写出规避代码

## 业务场景说明
本教程示例围绕 **LLMOps / LLM API 服务开发** 场景取材：模型响应解析、默认配置合并、请求参数转发、变长调用日志统计等。
（原需求中业务场景为占位符，此处依据本笔记所属的 LLMOps 学习目录合理假定；示例数据均为确定性数据，可整体替换。）

## 阅读约定
- Markdown 单元负责"为什么"（原理、要点、对比表），Code 单元负责"怎么验证"（全部真实执行，输出已嵌入）
- 代码注释风格：**编号知识点 + 逐行中文注释**
- 所有示例均为确定性输出（不依赖随机数；涉及集合处统一 `sorted()` 保证可复现）

## 源文档勘误预告（详见各章"修正说明"）
| 位置 | 源文档写法 | 问题 | 本笔记处理 |
|---|---|---|---|
| 第 3 章 嵌套+星号 | `a, *b, (c, d) = [1, 2, 3, 4, 5]`（注释称 c=4, d=5） | 末元素是 int，`(c,d)` 匹配失败，实际抛 `TypeError` | 改为 `[1, 2, 3, (4, 5)]`，反例单元实测验证 |
| 第 5.5 节 多次解包 | `def f(*args)` 却调用 `f(**{"a":1}, **{"b":2})` | 位置收集函数无法接收关键字参数，实际抛 `TypeError` | 拆成 `*args` 版与 `**kwargs` 版两个正确示例，反例单元实测验证 |
| 第 6.4 节 标题 | 标题写"字典推导 + 解包" | 内容未使用任何推导式 | 标题更正为"解包合并的进阶写法" |

## 第 0 章 前置知识：解包依赖的语言基础

**本章解决什么问题**：解包的合法性完全由"右侧是不是可迭代对象"决定。因此在进入正题前，先验证三个基础事实——可迭代对象与迭代器协议、赋值语句的目标列表、元组/字典的字面量规则。

| 前置概念 | 为什么解包需要它 |
|---|---|
| 可迭代对象（Iterable） | 解包右侧只要求"可迭代"，不要求是 list/tuple |
| 迭代器协议（`__iter__` / `__next__`） | 解包的底层执行机制：不断 `next()` 直到取完 |
| 赋值目标列表 | `a, b = ...` 的左侧在语言规范中叫 assignment target |
| 元组的逗号语法 | 函数"返回多值"实为返回元组，括号只是消歧 |
| 字典的 `keys()/values()/items()` | 决定解包字典时拿到的是键还是值 |

In [1]:
# [前置-1] 验证：可迭代对象 vs 迭代器协议
# 知识点①：实现 __iter__ 的对象就是可迭代对象，解包只认这个协议
# 知识点②：iter() 把可迭代对象转成迭代器，next() 逐个取值——这就是解包的底层动作
# 知识点③：迭代器是一次性的，取完即耗尽（后面生成器陷阱的理论根源）

from collections.abc import Iterable

nums = [1, 2, 3]                   # list 是可迭代对象
it = iter(nums)                    # 获得它的迭代器
print(next(it))                    # 1 —— 手动模拟解包时取出的第一个元素
print(next(it), next(it))          # 2 3 —— 迭代器被逐步消耗

print(isinstance(nums, Iterable))  # True  —— list 可迭代，可解包
print(isinstance(3, Iterable))      # False —— int 不可迭代
# 注意：a, b = 3 会报错，但 a, b = 3, 4 不会——因为 3, 4 先构成了元组

1
2 3
True
False


In [2]:
# [前置-2] 验证：元组逗号语法、函数返回多值、字典三个视图
# 知识点①：元组由逗号定义，括号只是可选的消歧符号
t1 = 1, 2, 3                       # 不加括号也是元组
t2 = (4,)                          # 单元素元组必须带逗号
t3 = (4)                           # 这不是元组，就是整数 4
print(type(t1).__name__, t1)       # tuple (1, 2, 3)
print(type(t2).__name__, type(t3).__name__)   # tuple int

# 知识点②：函数返回多值 = 隐式返回元组
def pair():
    return 1, 2                    # 等价于 return (1, 2)
p = pair()
print(type(p).__name__, p)          # tuple (1, 2)

# 知识点③：字典解包默认取"键"，.values() 才取值（Python 3.7+ 保证插入顺序，输出确定）
d = {"host": "localhost", "port": 8080}
k1, k2 = d                         # 直接解包 → 拿到键名
v1, v2 = d.values()                # 显式取值
print(k1, k2)                      # host port
print(v1, v2)                      # localhost 8080

tuple (1, 2, 3)
tuple int
tuple (1, 2)
host port
localhost 8080


## 第 1 章 基本解包

**本章解决什么问题**：把一个可迭代对象里的元素"拆开"赋给多个变量——这是所有后续语法的地基。

**要点列表**
1. 右侧可以是**任意可迭代对象**：list、tuple、str、range、生成器、字典、自定义可迭代类
2. 不带星号时，元素个数必须与左侧变量个数**严格相等**，否则 `ValueError`
3. 交换变量 `a, b = b, a` 的本质：右侧先构造元组 `(b, a)`，再解包回左侧
4. 函数返回多值即返回元组，解包让接收端写法自然

**等价写法对照**

| 写法 | 等价过程 |
|---|---|
| `a, b = b, a` | `tmp = (b, a)` 再 `a, b = tmp[0], tmp[1]` |
| `x, y = f()` | `t = f()` 再 `x, y = t[0], t[1]` |
| `p, q = "ab"` | `p, q = "ab"[0], "ab"[1]` |

In [3]:
# [1-1] 基本解包的五种形态
# 知识点①：list → 多变量，个数严格相等
a, b, c = [1, 2, 3]
print(a, b, c)                     # 1 2 3

# 知识点②：tuple → 多变量
x, y = (10, 20)
print(x, y)                        # 10 20

# 知识点③：str → 多变量（按字符拆分）
p, q = "ab"
print(p, q)                        # a b

# 知识点④：交换变量——右侧 b, a 先构造元组 (2, 1)，再解包给 a, b
a, b = 1, 2
a, b = b, a
print(a, b)                        # 2 1

# 知识点⑤：函数返回多值（本质是元组）→ 一次接收
def get_point():
    return 3, 4                    # 隐式返回 (3, 4)

x, y = get_point()
print(x, y)                        # 3 4

1 2 3
10 20
a b
2 1
3 4


In [4]:
# [1-2] 遍历时解包 + 调用端星号展开
# 知识点①：遍历"元素本身是序列"的容器，循环变量直接解包
points = [(1, 2), (3, 4), (5, 6)]
for x, y in points:                # 每轮把 (x, y) 从二元组中拆出
    print(x, y)                    # 1 2 / 3 4 / 5 6

# 知识点②：遍历字典键值对（LLM 配置表场景）
d = {"model": "gpt-4o-mini", "port": 8080}
for key, value in d.items():       # items() 逐轮产出 (key, value) 元组
    print(key, value)              # model gpt-4o-mini / port 8080

# 知识点③：enumerate 计数与解包组合（计数 + 嵌套解包）
for i, (x, y) in enumerate(points):
    print(i, x, y)                 # 0 1 2 / 1 3 4 / 2 5 6

# 知识点④：调用端 * 展开——把可迭代对象"铺平"成多个位置实参
print(*[1, 2, 3])                  # 1 2 3
print(*"abc")                      # a b c
print(*range(3))                   # 0 1 2

1 2
3 4
5 6
model gpt-4o-mini
port 8080
0 1 2
1 3 4
2 5 6
1 2 3
a b c
0 1 2


## 第 2 章 扩展解包：星号表达式

**本章解决什么问题**：当元素个数不确定时，用一个带 `*` 的变量"收集剩余元素"，从此解包不再要求数量严格相等。

**要点列表**
1. `*var` 收集剩余元素，收集结果**永远是 list**——即使右侧是元组、生成器
2. 星号变量最多一个，但位置任意：头部、中部、尾部
3. `*all_items, = [1, 2, 3]` 必须带**尾逗号**，否则 `*all_items` 是语法错误
4. 没有剩余元素时，星号变量得到空列表 `[]`，不报错
5. 解包生成器会**消耗**生成器——它是一次性的

**收集位置对照表**

| 表达式 | 结果 |
|---|---|
| `first, *rest = [1,2,3,4,5]` | first=1, rest=[2,3,4,5] |
| `*init, last = [1,2,3,4,5]` | init=[1,2,3,4], last=5 |
| `first, *middle, last = [1,2,3,4,5]` | first=1, middle=[2,3,4], last=5 |
| `*head, a, b = [1,2,3,4,5]` | head=[1,2,3], a=4, b=5 |

In [5]:
# [2-1] 星号收集的三种位置
# 知识点①：星号在中部——首尾固定，中间收集
first, *rest = [1, 2, 3, 4, 5]
print(first, rest)                 # 1 [2, 3, 4, 5]

# 知识点②：星号在尾部前——收集前面，固定最后一个
*init, last = [1, 2, 3, 4, 5]
print(init, last)                  # [1, 2, 3, 4] 5

# 知识点③：首、中、尾三段
first, *middle, last = [1, 2, 3, 4, 5]
print(first, middle, last)         # 1 [2, 3, 4] 5

# 知识点④：星号在头部——尾部固定两个
*head, a, b = [1, 2, 3, 4, 5]
print(head, a, b)                  # [1, 2, 3] 4 5

1 [2, 3, 4, 5]
[1, 2, 3, 4] 5
1 [2, 3, 4] 5
[1, 2, 3] 4 5


In [6]:
# [2-2] 边界情况：单个星号、空收集、字符串与生成器
# 知识点①：只有星号变量时，赋值必须带尾逗号，否则 *all_items 是语法错误
*all_items, = [1, 2, 3]
print(all_items)                   # [1, 2, 3]

# 知识点②：没有剩余元素 → 得到空列表，而不是报错
first, *rest = [1]
print(first, rest)                 # 1 []
*init, last = [1]
print(init, last)                  # [] 1

# 知识点③：星号变量永远是 list（用 type() 实证，即使右侧是元组）
a, *b = (1, 2, 3)
print(type(b).__name__, b)         # list [2, 3]

# 知识点④：字符串解包——单个字符 + 剩余字符列表
a, *b = "hello"
print(a, b)                        # h ['e', 'l', 'l', 'o']

# 知识点⑤：生成器解包——解包过程把它消耗殆尽
a, *b, c = (x for x in range(5))
print(a, b, c)                     # 0 [1, 2, 3] 4

[1, 2, 3]
1 []
[] 1
list [2, 3]
h ['e', 'l', 'l', 'o']
0 [1, 2, 3] 4


## 第 3 章 嵌套解包

**本章解决什么问题**：右侧是多层结构时，左侧用同样的嵌套形状逐层对应地拆——这是解析结构化数据（如 LLM 响应、CSV 行）的核心手段。

**要点列表**
1. 左侧的括号结构必须与右侧元素的内部结构逐层匹配
2. 嵌套目标可以是元组、列表等任意序列，混合类型亦可
3. 嵌套 + 星号可以混用：星号按"普通目标占掉的数量"计算剩余
4. 结构不匹配时尽早失败（`ValueError` / `TypeError`），比链式下标更安全——错误定位在赋值处

> **修正说明（源文档勘误）**：源文档写 `a, *b, (c, d) = [1, 2, 3, 4, 5]` 并注释输出 c=4、d=5。
> 该写法中 `(c, d)` 会匹配最后一个元素 `5`（int 不可再拆），实际抛 `TypeError`。正确写法是让末元素成为二元序列：`a, *b, (c, d) = [1, 2, 3, (4, 5)]`。两种写法都在下方单元实测验证。

In [7]:
# [3-1] 嵌套解包：左侧结构与右侧结构逐层对应
# 知识点①：括号嵌套对应元组嵌套
(a, b), c = (1, 2), 3
print(a, b, c)                     # 1 2 3

# 知识点②：完全一致的嵌套形状
(a, (b, c)) = (1, (2, 3))
print(a, b, c)                     # 1 2 3

# 知识点③：右侧可以是混合容器（list + tuple + list）
data = [1, (2, 3), [4, 5]]
a, (b, c), (d, e) = data
print(a, b, c, d, e)               # 1 2 3 4 5

# 知识点④：嵌套 + 星号混合使用（修正后的正确写法）
a, *b, (c, d) = [1, 2, 3, (4, 5)]  # 星号收集 [2, 3]，末元素 (4, 5) 再拆成 c=4, d=5
print(a, b, c, d)                  # 1 [2, 3] 4 5

1 2 3
1 2 3
1 2 3 4 5
1 [2, 3] 4 5


## 第 4 章 函数参数中的解包（定义端收集）

**本章解决什么问题**：函数如何"接收"数量不定的参数——`*args` 收集位置参数，`**kwargs` 收集关键字参数，这是转发/代理/装饰器模式的基础。

**参数顺序表（Python 3.8+ 完整语法）**

| 签名位置 | 语法 | 名称 | 传参方式 |
|---|---|---|---|
| `/` 之前 | `pos1, pos2, /` | 位置仅参数 | 只能按位置 |
| `/` 与 `*` 之间 | `normal` | 位置或关键字参数 | 两者皆可 |
| `*` 之后 | `kw_only` | 仅关键字参数 | 只能按关键字 |
| `*args` | 收集多余位置参数 | → 元组 |
| `**kwargs` | 收集多余关键字参数 | → 字典 |

**完整签名顺序**：`def f(pos_only, /, normal, *args, kw_only, **kwargs)`

**收集结果类型对照**

| 语法 | 收集结果类型 | 空调用时 |
|---|---|---|
| `*args` | `tuple` | `()` |
| `**kwargs` | `dict` | `{}` |

In [8]:
# [4-1] 函数定义端：*args 与 **kwargs
# 知识点①：*args 把多余位置参数收集成元组
def f_args(*args):
    print(args, type(args).__name__)   # 打印收集结果与类型
f_args(1, 2, 3)                        # (1, 2, 3) tuple
f_args()                               # () tuple

# 知识点②：**kwargs 把多余关键字参数收集成字典
def f_kw(**kwargs):
    print(kwargs, type(kwargs).__name__)
f_kw(a=1, b=2)                         # {'a': 1, 'b': 2} dict

# 知识点③：完整签名——普通参数 → *args → 仅关键字参数（带默认值）→ **kwargs
def f_full(a, b, *args, c, d=10, **kwargs):
    print(a, b, args, c, d, kwargs)
f_full(1, 2, 3, 4, c=5, e=6)           # 1 2 (3, 4) 5 10 {'e': 6}

# 知识点④：仅关键字参数（* 之后强制用关键字，提升可读性）
def f_kwonly(*, a, b):
    print(a, b)
f_kwonly(a=1, b=2)                     # 1 2 —— 必须写 a=、b=（f_kwonly(1, 2) 会 TypeError）

# 知识点⑤：位置仅参数 /（Python 3.8+，参数名不对外暴露）
def f_posonly(p1, p2, /, normal, *, kw_only):
    print(p1, p2, normal, kw_only)
f_posonly(1, 2, 3, kw_only=4)          # 1 2 3 4 —— p1/p2 只能按位置传

(1, 2, 3) tuple
() tuple
{'a': 1, 'b': 2} dict
1 2 (3, 4) 5 10 {'e': 6}
1 2
1 2 3 4


## 第 5 章 函数调用中的解包（调用端展开）

**本章解决什么问题**：与第 4 章正好对称——把已经装在容器里的值"拆回"成参数传给函数。定义端收集、调用端展开，`*` / `**` 是同一对符号的两个方向。

**定义端 vs 调用端**

| 方向 | 语法 | 作用 | 结果 |
|---|---|---|---|
| 定义端 | `def f(*args)` | 收集位置实参 | 元组 |
| 定义端 | `def f(**kwargs)` | 收集关键字实参 | 字典 |
| 调用端 | `f(*iterable)` | 展开可迭代对象 | 多个位置实参 |
| 调用端 | `f(**mapping)` | 展开映射 | 多个关键字实参 |

**要点**
1. `f(*nums)` 等价于把元素逐个按位置传入
2. `f(**d)` 要求字典的键**全是字符串**且与形参名匹配
3. `*` 与 `**` 可在同一调用中混用：`f(*args, **kwargs)`
4. Python 3.5+ 允许一次调用中**多次** `*` / `**` 解包，依次拼接合并
5. 不要对 set 使用 `*` 传位置参数——集合无序，映射到形参的顺序不确定

> **修正说明（源文档勘误）**：源文档 5.5 节中 `def f(*args)` 却调用 `f(**{"a": 1}, **{"b": 2})`。
> 该函数无法接收关键字参数，实际抛 `TypeError`。多次 `**` 解包要求函数定义含 `**kwargs`，下方代码已拆成两个正确示例，反例单元中实测验证。

In [9]:
# [5-1] 调用端解包：* 展开位置参数，** 展开关键字参数
# 知识点①：* 展开序列到位置参数
def add(a, b, c):
    return a + b + c
print(add(*[1, 2, 3]))             # 6 —— 等价 add(1, 2, 3)
print(add(*(1, 2, 3)))             # 6 —— 元组同样适用

# 知识点②：** 展开字典到关键字参数（键必须与形参名一致）
def greet(name, age):
    print(f"{name}, {age}")
profile = {"name": "Alice", "age": 30}
greet(**profile)                   # Alice, 30

# 知识点③：* 与 ** 同时使用（LLM SDK 常见转发写法）
def f4(a, b, c, d):
    print(a, b, c, d)
args = (1, 2)
kwargs = {"c": 3, "d": 4}
f4(*args, **kwargs)                # 1 2 3 4

# 知识点④：混合来源合并传参（Python 3.5+）
def f_any(*args, **kwargs):
    print(args, kwargs)
f_any(*[1, 2], **{"a": 3})         # (1, 2) {'a': 3}

# 知识点⑤：多次 * 解包依次拼接为一个元组
def f_pos(*args):
    print(args)
f_pos(*[1, 2], *[3, 4])            # (1, 2, 3, 4)

# 知识点⑥：多次 ** 解包合并为一个字典——函数必须含 **kwargs（见修正说明）
def f_kw2(**kwargs):
    print(kwargs)
f_kw2(**{"a": 1}, **{"b": 2})      # {'a': 1, 'b': 2}

6
6
Alice, 30
1 2 3 4
(1, 2) {'a': 3}
(1, 2, 3, 4)
{'a': 1, 'b': 2}


## 第 6 章 字典解包

**本章解决什么问题**：`{**d1, **d2}` 是 Python 中合并字典、覆盖默认配置的标准表达式写法——LLM 服务里"默认配置 + 用户覆盖"几乎都靠它。

**要点列表**
1. `{**d1, **d2}`：**后写的键覆盖先写的键**
2. 解包合并不修改原字典，产生新字典（表达式化、无副作用）
3. 解包 + 新键字面量可一步构造扩展字典
4. 调用端 `f(**{"a":1}, **{"b":2})` 同样遵循后覆盖前（键不冲突才合法）

> **修正说明（源文档勘误）**：源文档 6.4 节标题写"字典推导 + 解包"，但示例内容并未使用任何推导式，实为解包合并的进阶写法。本笔记已更正该节标题，避免误导。

In [10]:
# [6-1] 字典解包：合并、覆盖、构造
# 知识点①：{**d1, **d2} 合并字典，后写者覆盖先写者
d1 = {"a": 1, "b": 2}
d2 = {"b": 3, "c": 4}
print({**d1, **d2})                # {'a': 1, 'b': 3, 'c': 4} —— d2 的 b 覆盖 d1 的 b

# 知识点②：默认配置 + 用户覆盖（LLM 请求参数场景），不污染原字典
defaults = {"model": "gpt-4o-mini", "temperature": 0.7, "timeout": 30}
user = {"temperature": 0.2}
config = {**defaults, **user}
print(config)                      # temperature 被 0.2 覆盖
print(defaults)                    # 原字典未被修改，仍是 0.7

# 知识点③：解包 + 新键字面量，一步构造扩展字典
d = {"api_key": "sk-xxx"}
new_d = {**d, "base_url": "https://api.example.com", "timeout": 60}
print(new_d)                       # {'api_key': 'sk-xxx', 'base_url': ..., 'timeout': 60}

# 知识点④：多个字典 + 新键一次混合合并（更正后的 6.4 内容）
merged = {**{"a": 1}, **{"b": 2}, "c": 3}
print(merged)                      # {'a': 1, 'b': 2, 'c': 3}

# 知识点⑤：调用端合并多个 ** 再传给 **kwargs
def f_opts(**kwargs):
    print(kwargs)
f_opts(**{"stream": True}, **{"max_tokens": 1024})   # {'stream': True, 'max_tokens': 1024}

{'a': 1, 'b': 3, 'c': 4}
{'model': 'gpt-4o-mini', 'temperature': 0.2, 'timeout': 30}
{'model': 'gpt-4o-mini', 'temperature': 0.7, 'timeout': 30}
{'api_key': 'sk-xxx', 'base_url': 'https://api.example.com', 'timeout': 60}
{'a': 1, 'b': 2, 'c': 3}
{'stream': True, 'max_tokens': 1024}


## 第 7 章 序列解包与字典解包对比

**本章解决什么问题**：`*` 与 `**` 在不同容器语境下的完整速查，并验证四种容器的合并写法互相等价。

| 语法 | 用途 |
|---|---|
| `*iterable`（调用中） | 解包可迭代对象为**位置参数** |
| `**mapping`（调用中） | 解包字典为**关键字参数** |
| `{**d1, **d2}` | 合并字典 |
| `[*a, *b]` | 合并列表 |
| `(*a, *b)` | 合并元组 |
| `{*a, *b}` | 合并集合 |

In [11]:
# [7-1] 四种容器的解包合并写法对比验证
a = [1, 2]
b = [3, 4]

# 知识点①：合并列表
print([*a, *b])                    # [1, 2, 3, 4]
# 知识点②：合并元组
print((*a, *b))                    # (1, 2, 3, 4)
# 知识点③：合并集合（集合无序，用 sorted 保证输出确定）
print(sorted({*a, *b}))            # [1, 2, 3, 4]
# 知识点④：合并字典
d1 = {"a": 1}
d2 = {"b": 2}
print({**d1, **d2})                # {'a': 1, 'b': 2}

# 知识点⑤：等价性验证——* 合并与 + 拼接结果一致
print([*a, *b] == [*a] + [*b])     # True

# 知识点⑥：Python 3.9+ 字典 | 运算符与 {**d1, **d2} 等价
print(d1 | d2 == {**d1, **d2})     # True

[1, 2, 3, 4]
(1, 2, 3, 4)
[1, 2, 3, 4]
{'a': 1, 'b': 2}
True
True


## 第 8 章 解包协议原理

**本章解决什么问题**：解包到底"认"什么？答案：只认迭代器协议（`__iter__`），不认具体类型——这决定了哪些对象能解包、星号为何必得 list、数量不匹配为何报 `ValueError`。

**要点列表**
1. 右侧对象只要有 `__iter__` 就能解包——与它是 list、生成器还是自定义类无关
2. 解包字典默认迭代**键**（字典本身就是"键的迭代器"）
3. 带星号解包时，剩余元素被逐个放进新建的 list——所以星号变量必是 list
4. 元素数量不匹配 → `ValueError`（too many / not enough），可用星号兜底规避

In [12]:
# [8-1] 协议验证：只要实现 __iter__ 就能被解包
# 知识点①：range / str / 生成器 都能直接解包
a, b, c = range(3)
print(a, b, c)                     # 0 1 2
a, b, c = "abc"
print(a, b, c)                     # a b c
a, b, c = (x for x in range(3))    # 生成器被解包即消耗
print(a, b, c)                     # 0 1 2

# 知识点②：解包字典默认拿到键
k1, k2 = {"model": "m1", "port": 8080}
print(k1, k2)                      # model port

# 知识点③：自定义类只要实现 __iter__ 即可解包
class MyIterable:
    def __iter__(self):            # 实现迭代协议即可，不要求是任何内置类型
        yield 1
        yield 2
        yield 3

a, b, c = MyIterable()
print(a, b, c)                     # 1 2 3

# 知识点④：带星号解包自定义可迭代对象 → 剩余进 list
a, *b = MyIterable()
print(a, b, type(b).__name__)      # 1 [2, 3] list

0 1 2
a b c
0 1 2
model port
1 2 3
1 [2, 3] list


In [13]:
# [8-2] 验证：数量不匹配的两种 ValueError（try/except 捕获展示，保证单元可复现）
# 知识点①：3 个元素塞给 2 个变量 → too many
try:
    a, b = [1, 2, 3]
except ValueError as e:
    print(type(e).__name__, "-", e)

# 知识点②：2 个元素分给 3 个变量 → not enough
try:
    a, b, c = [1, 2]
except ValueError as e:
    print(type(e).__name__, "-", e)

# 知识点③：星号可兜底任意变长，规避以上两类错误
a, *b = [1, 2, 3]
print(a, b)                        # 1 [2, 3]

ValueError - too many values to unpack (expected 2)
ValueError - not enough values to unpack (expected 3, got 2)
1 [2, 3]


## 反例单元：错误写法 → 实际触发报错 → 正确写法对比

**本章解决什么问题**：与其背规则，不如亲眼看到报错。以下每个反例都**真实触发异常**（用 try/except 捕获打印，保证整本 Notebook 0 报错单元），随后给出正确写法并对比输出差异。

In [14]:
# [反例-1] 元素数量不匹配
# ✗ 错误写法：2 个变量接 3 个元素 → ValueError
try:
    a, b = [1, 2, 3]
except ValueError as e:
    print("错误写法报错:", e)
# ✓ 正确写法：星号兜底剩余元素
a, *b = [1, 2, 3]
print("正确写法输出:", a, b)       # 1 [2, 3]

错误写法报错: too many values to unpack (expected 2)
正确写法输出: 1 [2, 3]


In [15]:
# [反例-2] 嵌套结构不匹配（源文档勘误实测）
# ✗ 源文档写法：末元素是 int 5，无法再拆成 (c, d) → TypeError
try:
    a, *b, (c, d) = [1, 2, 3, 4, 5]
except TypeError as e:
    print("错误写法报错:", e)
# ✓ 正确写法：让末元素成为二元序列
a, *b, (c, d) = [1, 2, 3, (4, 5)]
print("正确写法输出:", a, b, c, d)  # 1 [2, 3] 4 5

错误写法报错: cannot unpack non-iterable int object
正确写法输出: 1 [2, 3] 4 5


In [16]:
# [反例-3] ** 解包的键不是字符串
def f_kw(**kwargs):
    print(kwargs)
# ✗ 错误写法：int 键不是合法关键字名 → TypeError
try:
    f_kw(**{1: "a"})
except TypeError as e:
    print("错误写法报错:", e)
# ✓ 正确写法：键必须是字符串
f_kw(**{"1": "a"})                 # {'1': 'a'}

错误写法报错: keywords must be strings
{'1': 'a'}


In [17]:
# [反例-4] 重复关键字参数
def f2(a, b):
    print(a, b)
# ✗ 错误写法：两个来源提供同名键 → TypeError
try:
    f2(**{"a": 1}, **{"a": 2})
except TypeError as e:
    print("错误写法报错:", e)
# ✓ 正确写法：键不冲突即可合并
f2(**{"a": 1}, **{"b": 2})         # 1 2

错误写法报错: __main__.f2() got multiple values for keyword argument 'a'
1 2


In [18]:
# [反例-5] 生成器解包后耗尽（源文档 5.5 勘误实测）
# ✗ 错误写法一：位置收集函数接收多次 ** 解包 → TypeError
def f_pos_only(*args):
    print(args)
try:
    f_pos_only(**{"a": 1}, **{"b": 2})
except TypeError as e:
    print("错误写法一报错:", e)
# ✓ 正确写法：多次 ** 解包要求函数定义含 **kwargs
def f_both(**kwargs):
    print(kwargs)
f_both(**{"a": 1}, **{"b": 2})     # {'a': 1, 'b': 2}

# ✗ 错误写法二：生成器第二次解包 → ValueError（已耗尽）
g = (x for x in range(3))
first_unpack = next(g), next(g), next(g)
print("生成器手动取值:", first_unpack)
try:
    a, b, c = g                    # 生成器已耗尽，无值可给
except ValueError as e:
    print("错误写法二报错:", e)
# ✓ 正确写法：需要多次消费时先物化为 list
nums = list(range(3))
x1, x2, x3 = nums
x1b, x2b, x3b = nums               # list 可重复解包
print("list 可重复解包:", x1, x2, x3, x1b, x2b, x3b)

错误写法一报错: f_pos_only() got an unexpected keyword argument 'a'
{'a': 1, 'b': 2}
生成器手动取值: (0, 1, 2)
错误写法二报错: not enough values to unpack (expected 3, got 0)
list 可重复解包: 0 1 2 0 1 2


In [19]:
# [反例-6] 星号不能单独作为赋值目标（语法错误，编译期即可发现）
# ✗ 错误写法：* = [1, 2, 3] → SyntaxError（用 compile() 在运行期复现编译错误）
try:
    compile("* = [1, 2, 3]", "<反例>", "exec")
except SyntaxError as e:
    print("错误写法报错: SyntaxError -", e.msg)
# ✓ 正确写法：星号必须绑定一个变量名
*items, = [1, 2, 3]
print("正确写法输出:", items)      # [1, 2, 3]

错误写法报错: SyntaxError - invalid syntax
正确写法输出: [1, 2, 3]


## 实用示例集：LLM API 开发中的 5 个高频模式

**本章解决什么问题**：把解包语法落到工程高频场景。下表给出各模式的适用场景与注意点，随后每个模式一个可运行单元。

| # | 模式 | 适用场景 | 注意点 |
|---|---|---|---|
| 1 | 响应/嵌套数据解析 | 解析 LLM 响应 choices、结构化行 | 结构不符时尽早失败，错误定位准确 |
| 2 | 配置合并 `{**defaults, **user}` | 默认参数 + 租户/用户覆盖 | 浅拷贝语义：嵌套 dict 仍是引用 |
| 3 | 参数转发 `wrapper(*args, **kwargs)` | 装饰器、代理、SDK 重试封装 | 保持签名透明；functools.wraps 保元信息 |
| 4 | 变长记录统计 `name, *scores` | 每条记录字段数不一致的日志 | 星号变量必是 list，空记录得到 `[]` 要防除零 |
| 5 | 去重保序 + 行拆分 | API 端点去重、CSV 行解析 | `dict.fromkeys` 保序去重；`split` 返回值可直接解包 |

In [20]:
# [模式-1] 响应/嵌套数据解析（第 1、3 章组合）
# 场景：解析 LLM 响应的 choices 列表，元素是 (index, finish_reason, (role, content)) 的嵌套结构
resp_choices = [
    (0, "stop", ("assistant", "解包是双向工具")),
    (1, "length", ("assistant", "输出被截断")),
]
# 知识点：for 变量区直接写嵌套解包目标，一步拆到底
for idx, finish, (role, content) in resp_choices:
    print(f"[{idx}] {role}: {content} (finish={finish})")
# 输出：
# [0] assistant: 解包是双向工具 (finish=stop)
# [1] assistant: 输出被截断 (finish=length)

[0] assistant: 解包是双向工具 (finish=stop)
[1] assistant: 输出被截断 (finish=length)


In [21]:
# [模式-2] 配置合并：默认配置 + 用户覆盖（第 6 章）
# 场景：LLM 请求参数 = 平台默认值 ⊕ 租户覆盖值
DEFAULTS = {"model": "gpt-4o-mini", "temperature": 0.7, "max_tokens": 512}
tenant = {"temperature": 0.2, "max_tokens": 1024}
request_config = {**DEFAULTS, **tenant}      # 后写覆盖先写
print(request_config)                        # {'model': 'gpt-4o-mini', 'temperature': 0.2, 'max_tokens': 1024}

# 注意点实证：解包合并是"浅拷贝"——嵌套 dict 仍共享引用
d1 = {"retry": {"max": 3}}                   # 嵌套字典
d2 = {**d1}                                  # 浅拷贝
d2["retry"]["max"] = 5
print(d1["retry"]["max"])                    # 5 —— 原 dict 内层被一并改动

{'model': 'gpt-4o-mini', 'temperature': 0.2, 'max_tokens': 1024}
5


In [22]:
# [模式-3] 参数转发：装饰器/代理（第 4、5 章组合）
# 场景：给任意 SDK 调用包一层超时日志，不关心原函数签名
from functools import wraps

def log_call(fn):
    @wraps(fn)                               # 保留原函数元信息
    def wrapper(*args, **kwargs):            # 定义端：收集任意参数
        result = fn(*args, **kwargs)         # 调用端：原样展开转发
        return result
    return wrapper

def embed(text, model="text-embedding-3", **extra):
    return {"text": text, "model": model, "extra": extra}

embed_logged = log_call(embed)
print(embed_logged("hello", model="bge-m3", user="u1"))
# {'text': 'hello', 'model': 'bge-m3', 'extra': {'user': 'u1'}}

{'text': 'hello', 'model': 'bge-m3', 'extra': {'user': 'u1'}}


In [23]:
# [模式-4] 变长记录统计（第 2 章）
# 场景：不同 API 的调用日志字段数不一致，用星号收集 token 明细
logs = [
    ("chat", 120, 64),                       # 3 列
    ("embed", 300),                          # 2 列
    ("rerank", 50, 20, 10),                  # 4 列
]
for api_name, *tokens in logs:               # 星号兜底任意长度
    total = sum(tokens)
    print(f"{api_name}: tokens={tokens} total={total}")
# 输出：
# chat: tokens=[120, 64] total=184
# embed: tokens=[300] total=300
# rerank: tokens=[50, 20, 10] total=80

# 注意点：空明细得到 []，求平均要防除零
name, *scores = ("webhook",)
avg = sum(scores) / len(scores) if scores else 0.0
print(f"{name}: scores={scores} avg={avg}")  # webhook: scores=[] avg=0.0

chat: tokens=[120, 64] total=184
embed: tokens=[300] total=300
rerank: tokens=[50, 20, 10] total=80
webhook: scores=[] avg=0.0


In [24]:
# [模式-5] 去重保序 + 行拆分（第 1、7 章）
# 场景①：URL 端点列表去重且保持首次出现顺序
endpoints = ["/chat", "/embed", "/chat", "/rerank", "/embed"]
unique_endpoints = [*dict.fromkeys(endpoints)]   # dict.fromkeys 保序去重，星号物化为 list
print(unique_endpoints)                          # ['/chat', '/embed', '/rerank']

# 场景②：CSV 行拆分——split() 返回 list，直接解包
line = "Alice,30,alice@example.com"
name, age, email = line.split(",")               # 逐列拆到独立变量
print(name, age, email)                          # Alice 30 alice@example.com

# 注意点：列数不符会 ValueError，用星号兜底更稳健
line2 = "Bob,25"                                 # 缺 email 列
name2, *rest2 = line2.split(",")
print(name2, rest2)                              # Bob ['25']

['/chat', '/embed', '/rerank']
Alice 30 alice@example.com
Bob ['25']


## 内置 / 生态对照：与解包相关的语言及标准库能力

| 能力 | 版本 | 所属 | 典型用法 | 与解包的关系 |
|---|---|---|---|---|
| `enumerate()` | — | 内置 | `for i, x in enumerate(xs)` | 遍历解包的计数版 |
| `zip()` / `zip(*rows)` | — | 内置 | 行列转置 | `zip(*rows)` 本质是调用端 `*` 解包 |
| `dict.fromkeys()` | — | 内置 | `[*dict.fromkeys(xs)]` | 配合星号物化实现保序去重 |
| `dict.values()/items()` | — | 内置 | `a, b = d.values()` | 字典解包取值的正确姿势 |
| `{**d1, **d2}` | 3.5+ | 语言 | 字典合并 | 调用端 `**` 的字面量语境版 |
| `d1 \| d2` | 3.9+ | 语言 | 字典合并 | 与 `{**d1, **d2}` 等价（右侧覆盖） |
| `[*a, *b]` / `(*a, *b)` | 3.5+ | 语言 | 序列合并 | 星号在字面量内多次展开 |
| 位置仅参数 `/` | 3.8+ | 语言 | `def f(a, b, /)` | 限定只能按位置传（配合解包使用） |
| `match-case` 序列模式 | 3.10+ | 语言 | `case [first, *rest]` | 解包的"模式匹配版"，可条件化 |
| `typing.NamedTuple` / `dataclass` | — | 标准库 | 结构化记录 | 字段名替代位置解包，更可读 |
| `itertools.chain(*iters)` | — | 标准库 | 惰性拼接 | `[*a, *b]` 的惰性等价物（省内存） |
| `pydantic.BaseModel(**data)` | 生态 | pydantic | 请求模型构造 | `**` 解包注入字段并做校验 |

In [25]:
# [对照-1] 表格中关键等价关系的验证
d1 = {"a": 1, "b": 2}
d2 = {"b": 3, "c": 4}

# 知识点①：| 运算符与 {**d1, **d2} 完全等价（Python 3.9+）
print(d1 | d2 == {**d1, **d2})                  # True

# 知识点②：zip(*rows) 用的就是调用端 * 解包（行列转置）
rows = [(1, 2, 3), (4, 5, 6)]
cols = list(zip(*rows))                        # *rows 把两个元组展开成两个位置实参
print(cols)                                    # [(1, 4), (2, 5), (3, 6)]

# 知识点③：match-case 是解包的模式匹配版（Python 3.10+）
def parse_token_usage(data):
    match data:
        case [_, *rest] if rest:               # 首元素丢弃，剩余非空才收集
            return sum(rest)
        case _:
            return 0
print(parse_token_usage([0, 120, 64]))         # 184
print(parse_token_usage([0]))                   # 0

True
[(1, 4), (2, 5), (3, 6)]
184
0


## 综合实战：LLM API 调用管道中的解包组合拳

**本章解决什么问题**：把前面 8 章的知识点串成一条贴近 LLMOps 业务的主线——**组装请求 → 转发调用 → 解析响应 → 统计日志**。

### 设计取舍

| 决策点 | 选择 | 理由（对比备选方案） |
|---|---|---|
| 配置合并 | `{**DEFAULTS, **overrides}` 表达式 | 相比 `d = defaults.copy(); d.update(overrides)`：不修改原字典、一行完成、可内联在实参里 |
| 参数转发 | `wrapper(*args, **kwargs)` | 相比逐个枚举参数：对任意签名透明，装饰器/重试逻辑零耦合于具体函数 |
| 响应解析 | 嵌套解包 `for idx, finish, (role, content) in ...` | 相比链式下标 `choice[2][1]`：结构不符时**在赋值处立即失败**，错误定位清晰；下标写法错位后静默产出错误数据 |
| usage 提取 | `usage.values()` 解包 | 相比逐键取值：一行完成；前提是 dict 3.7+ 保插入序——字段固定场景可接受 |
| 变长日志 | `api_name, *tokens` | 相比 `len` 判断分支：一条语句处理任意列数 |

**注意点**：本示例的所有"模拟 SDK"均为确定性返回，保证整本 Notebook 每次执行输出一致。

In [26]:
# [实战-1] LLM API 调用管道：解包知识综合运用（确定性模拟，可整体复现）
# ========== 第 1 步：组装请求配置（第 6 章：字典解包合并，用户覆盖默认） ==========
DEFAULTS = {"model": "gpt-4o-mini", "temperature": 0.7, "max_tokens": 512}
tenant_overrides = {"temperature": 0.2, "max_tokens": 1024}
request_config = {**DEFAULTS, **tenant_overrides}      # 覆盖合并，不改原字典
print("请求配置:", request_config)

# ========== 第 2 步：参数转发装饰器（第 4、5 章：定义端收集 + 调用端展开） ==========
from functools import wraps

def with_retry(fn, retries=2):
    @wraps(fn)
    def wrapper(*args, **kwargs):                      # 定义端：收集任意签名
        last_exc = None
        for attempt in range(retries + 1):
            try:
                return fn(*args, **kwargs)            # 调用端：原样展开转发
            except RuntimeError as e:                # 模拟可重试的瞬时错误
                last_exc = e
        raise last_exc
    return wrapper

# ========== 第 3 步：模拟 SDK（确定性返回，记录调用次数） ==========
_call_count = 0
def mock_chat_call(prompt, **params):                 # **kwargs 接收全部配置参数
    global _call_count
    _call_count += 1
    return {
        "id": f"chatcmpl-{_call_count:03d}",
        "prompt": prompt,
        "usage": {"prompt_tokens": 128, "completion_tokens": 64},
        "choices": [(0, "stop", ("assistant", "解包是双向工具"))],
        "request_params": params,                    # 回显参数，验证 ** 展开正确
    }

chat = with_retry(mock_chat_call)
resp = chat("解释解包", **request_config)             # 第 5 章：** 解包展开为关键字实参
print("请求参数已透传:", resp["request_params"]["temperature"] == 0.2 and
      resp["request_params"]["model"] == "gpt-4o-mini")   # True

# ========== 第 4 步：解析响应（第 1、3 章：解包 + 嵌套解包） ==========
prompt_tokens, completion_tokens = resp["usage"].values()   # dict 保插入序，顺序解包
print(f"token 用量: prompt={prompt_tokens}, completion={completion_tokens}")
for idx, finish, (role, content) in resp["choices"]: # 嵌套解包一步拆到底
    print(f"[{idx}] {role}: {content} (finish={finish})")

# ========== 第 5 步：变长调用日志统计（第 2 章：星号收集） ==========
logs = [
    ("chat", _call_count, 128, 64),                   # 调用次数 + token 明细
    ("embed", 3, 300),
    ("rerank", 1, 50, 20, 10),
]
print("API 调用统计:")
for api_name, count, *tokens in logs:                 # 星号兜底任意列数
    print(f"  {api_name}: calls={count}, tokens={tokens}, total={sum(tokens)}")

请求配置: {'model': 'gpt-4o-mini', 'temperature': 0.2, 'max_tokens': 1024}
请求参数已透传: True
token 用量: prompt=128, completion=64
[0] assistant: 解包是双向工具 (finish=stop)
API 调用统计:
  chat: calls=1, tokens=[128, 64], total=192
  embed: calls=3, tokens=[300], total=300
  rerank: calls=1, tokens=[50, 20, 10], total=80


## 常见陷阱：陷阱 / 症状 / 解法汇总表

**本章解决什么问题**：把散落各章的易错点收敛成一张速查表；每个陷阱在前文反例单元或下方单元均有可运行的复现代码。

| # | 陷阱 | 症状（报错/表现） | 解法 | 复现位置 |
|---|---|---|---|---|
| 1 | 元素数量不匹配 | `ValueError: too many / not enough values` | 星号变量兜底 | [反例-1] |
| 2 | 嵌套结构与右侧不匹配 | `TypeError: cannot unpack non-iterable int` | 让左侧形状逐层对应右侧 | [反例-2] |
| 3 | `**` 的键不是字符串 | `TypeError: keywords must be strings` | 保证映射键为 str | [反例-3] |
| 4 | 重复关键字参数 | `TypeError: got multiple values for keyword` | 多来源键不冲突再合并 | [反例-4] |
| 5 | `*args` 函数接收 `**` 解包 | `TypeError: unexpected keyword argument` | 函数定义补 `**kwargs` | [反例-5] |
| 6 | 生成器二次解包 | `ValueError: not enough values`（已耗尽） | 先物化为 list/tuple | [反例-5] |
| 7 | `*` 单独作赋值目标 | `SyntaxError: starred assignment target must be...` | 星号必须绑定变量名 | [反例-6] |
| 8 | 星号变量类型想当然 | 以为是 tuple，实际是 **list** | 记住：星号收集永远是 list | 下方 [陷阱-8] |
| 9 | 解包字典默认取键 | 变量拿到的是 'host' 而非 'localhost' | `.values()` / `.items()` | 下方 [陷阱-9] |
| 10 | `_` 被多次赋值覆盖 | `_` 只保留最后一次的值 | 它只是普通约定变量 | 下方 [陷阱-10] |
| 11 | 集合 `*` 传位置参数 | 实参映射顺序不确定 | 改用序列或关键字传参 | 下方 [陷阱-11] |

In [27]:
# [陷阱-8] 星号变量总是 list（即使右侧是元组/字符串/生成器）
a, *b = (1, 2, 3)                  # 右侧是元组
print(type(b).__name__)            # list —— 不是 tuple！
*head, tail = "hello"              # 右侧是字符串
print(type(head).__name__, head)   # list ['h', 'e', 'l', 'l']
c, *d, e = (x for x in range(4))   # 右侧是生成器
print(type(d).__name__, d)         # list [1, 2]

list
list ['h', 'e', 'l', 'l']
list [1, 2]


In [28]:
# [陷阱-9] 解包字典默认取键
# 症状：想拿值，结果拿到的是键名
d = {"host": "localhost", "port": 8080}
a, b = d
print(a, b)                        # host port —— 拿到的是键！
# 解法：显式取值视图
a, b = d.values()
print(a, b)                        # localhost 8080

host port
localhost 8080


In [29]:
# [陷阱-10] _ 只是约定，不是"丢弃语义"
_, _, c = 1, 2, 3
print(c)                           # 3
print(_)                           # 3 —— _ 是普通变量，最后一次赋值覆盖前次
# 交互式环境中 _ 还保存上一次表达式的结果；生产代码不要依赖 _ 的内容
# 解法：需要明确丢弃多个值时用不同变量名或星号收集
first, *_ = [10, 20, 30, 40]
print(first)                       # 10

3
2
10


In [30]:
# [陷阱-11] 集合无序，不要用 * 映射到位置参数
# 症状：f(*{1, 2, 3}) 的实参顺序不确定（不同实现/不同时刻可能不同）
def which(first, second, third):
    print("first =", first)       # 三个形参谁接到 1 谁接到 3 无法保证
s = {1, 2, 3}
print("集合内部顺序(不保证):", s)
# 解法：传位置参数前先 sorted/list 物化，或直接用关键字参数
which(*sorted(s))                 # first = 1 —— 排序后顺序确定

集合内部顺序(不保证): {1, 2, 3}
first = 1


## 总结

### 通用代码模板（可直接复制）

```python
# ---- 赋值端 ----
a, b = 1, 2                          # 基本解包
a, b = b, a                          # 交换
first, *rest = items                 # 收集剩余（必得 list）
first, *mid, last = items            # 首中尾三段
(a, b), c = (1, 2), 3               # 嵌套解包
*all_items, = items                  # 全收集（注意尾逗号）
_, keep, _ = row                    # 丢弃无关值（_ 是普通变量）

# ---- 函数定义端（收集） ----
def f(pos, /, normal, *args, kw_only, **kwargs): ...
#        ↑位置仅  ↑普通   ↑元组    ↑仅关键字  ↑字典

# ---- 函数调用端（展开） ----
result = f(*args, **kwargs)          # 转发/代理标准写法
result = f(*[1, 2], *[3, 4])         # 多次 * 拼接（3.5+）
result = f(**{"a": 1}, **{"b": 2})   # 多次 ** 合并（3.5+，需 **kwargs）

# ---- 容器合并 ----
merged_list  = [*a, *b]              # 列表
merged_tuple = (*a, *b)              # 元组
merged_set   = {*a, *b}              # 集合
merged_dict  = {**d1, **d2}          # 字典（后覆盖前）
merged_dict9 = d1 | d2               # 字典（3.9+，等价）
unique_keep_order = [*dict.fromkeys(items)]   # 去重保序
```

### 记忆口诀

1. **右侧可迭即可拆，数量不符星号来**
2. **星号收集必成列表，右边是啥都不改**
3. **定义收集、调用展开，`*` 管位置、`**` 管键**
4. **字典解包默认拿键，要拿值就 `.values()`**
5. **生成器解包一次过，要用两次先物化**
6. **`**` 键必须字符串，关键字重复必报错**

### 自测练习（参考实现在下方单元）

1. 不用临时变量交换 `a=3, b=8`，并验证交换成功
2. 写函数 `min_max(nums)` 一次返回最小值和最大值，并用解包接收
3. 从 `"Alice,30,alice@example.com"` 中拆出姓名、年龄、邮箱；再写一个对缺列稳健的版本
4. 合并 `DEFAULTS` 与 `user` 两个配置字典，要求用户键覆盖默认键且不改原字典，并验证
5. 写一个转发参数并打印调用日志的装饰器，用在一个接收混合参数的函数上
6. 解析 `[("Alice", (1990, 5, 20)), ("Bob", (1985, 11, 3))]`，打印 `姓名: YYYY-MM-DD`

In [31]:
# [练习-参考实现] 自测练习 1–3
# 练习 1：交换变量并验证
a, b = 3, 8
a, b = b, a
print("练习1:", a, b, a == 8 and b == 3)     # 8 3 True

# 练习 2：函数返回多值 + 解包接收
def min_max(nums):
    return min(nums), max(nums)              # 返回元组
low, high = min_max([3, 1, 4, 1, 5])
print("练习2:", low, high)                   # 1 5

# 练习 3：CSV 行拆分（严格版 + 稳健版）
line = "Alice,30,alice@example.com"
name, age, email = line.split(",")            # 严格版：列数必须正好 3
print("练习3a:", name, age, email)            # Alice 30 alice@example.com
line2 = "Bob,25"                              # 缺列数据
name2, *rest2 = line2.split(",")              # 稳健版：星号兜底
print("练习3b:", name2, rest2)                 # Bob ['25']

练习1: 8 3 True
练习2: 1 5
练习3a: Alice 30 alice@example.com
练习3b: Bob ['25']


In [32]:
# [练习-参考实现] 自测练习 4–6
# 练习 4：配置合并，用户覆盖默认，且不改原字典
DEFAULTS = {"model": "gpt-4o-mini", "temperature": 0.7, "stream": False}
user = {"temperature": 0.1}
config = {**DEFAULTS, **user}
print("练习4:", config["temperature"] == 0.1, DEFAULTS["temperature"] == 0.7)   # True True

# 练习 5：参数转发装饰器（定义端收集 + 调用端展开 + wraps）
from functools import wraps

def log_call(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        print(f"  调用 {fn.__name__}, 位置参数={args}, 关键字参数={kwargs}")
        return fn(*args, **kwargs)            # 原样转发
    return wrapper

@log_call
def create_user(name, age, email, *, role="user"):
    return {"name": name, "age": age, "email": email, "role": role}

u = create_user("Alice", 30, "a@example.com", role="admin")
print("练习5:", u)

# 练习 6：解析嵌套生日数据
data = [
    ("Alice", (1990, 5, 20)),
    ("Bob", (1985, 11, 3)),
]
for name, (year, month, day) in data:         # 嵌套解包一步拆出年月日
    print(f"练习6: {name}: {year}-{month:02d}-{day:02d}")
# 练习6: Alice: 1990-05-20
# 练习6: Bob: 1985-11-03

练习4: True True
  调用 create_user, 位置参数=('Alice', 30, 'a@example.com'), 关键字参数={'role': 'admin'}
练习5: {'name': 'Alice', 'age': 30, 'email': 'a@example.com', 'role': 'admin'}
练习6: Alice: 1990-05-20
练习6: Bob: 1985-11-03


---
## 结语

> **解包是"把容器拆成独立变量，或把独立值组装成参数"的双向工具：`*` 管位置，`**` 管关键字。**

下一步建议：把 [实战-1] 的管道示例接入你真实的 LLM SDK（OpenAI / LangChain），观察 `**kwargs` 转发与响应解析在真实返回结构上的表现。